<a href="https://colab.research.google.com/github/AlejandroBlandonV/Analisis-American-Development-Bank/blob/main/Analisis-American-Development-Bank%20Movilidad%20y%20PIB%20en%20LATAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🧩 Paso 1: Cargar y explorar

Antes de limpiar o combinar los datos, es necesario **familiarizarte con la estructura de ambos datasets**.
En esta etapa, validarás que los archivos se carguen correctamente, conocerás sus columnas y tipos de datos, y detectarás posibles inconsistencias.

### 1.1 Carga de datos y vista rápida

**🎯Objetivo:**
Importar las librerías necesarias, cargar los archivos CSV en DataFrames y realizar una revisión preliminar para entender su contenido.

**Instrucciones:**
- Importa las librerías `pandas`, `numpy`, `seaborn` y `matplotlib.pyplot`.
- Carga los archivos usando `pd.read_csv()`:
  - `'/datasets/tomtom_traffic.csv'`
  - `/datasets/oecd_city_economy.csv` `.
- Guarda los DataFrames en las variables `traffic` y `eco`.
- Muestra las primeras 5 filas de cada DataFrame.


In [ ]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
traffic = pd.read_csv('/datasets/tomtom_traffic.csv')
eco = pd.read_csv('/datasets/oecd_city_economy.csv')


FileNotFoundError: [Errno 2] No such file or directory: '/datasets/tomtom_traffic.csv'

In [ ]:
# mostrar las primeras 5 filas de traffic
traffic.head()

In [ ]:
# mostrar las primeras 5 filas de eco
eco.head()

**Tip:** Si no usas `print()` la tabla se vera mejor.


---

## 🧩Paso 2: Explorar, limpiar y preparar los datos

Antes de combinar los datasets, inspecciona su estructura, tipos de datos, columnas y valores faltantes.
Anota las columnas que necesiten limpieza y luego estandariza los nombres de columnas.

### 2.1 Explorar la estructura y tipos de datos

**🎯Objetivo:**
Identificar columnas con tipos incorrectos, distribución y nulos, anotar las columnas que requieren conversión.

**Instrucciones:**

- Usa `.info()` para conocer la estructura de ambos DataFrames.
- Muestra los primeros 3 renglones de cada DF.
- Identifica si los detalles de cada DF estan bien o si requieren correcciones y escribe tus conclusiones en el bloque Markdown.
  - ¿Hay columnas que requieren conversión?¿ Cuáles son? ¿Que tipo de dato ienen y cuál deberían de tener?
  - ¿Hay datos ausentes en alguna columna?


In [ ]:
# Examinar la estructura de traffic
traffic.info()
traffic.tail(3)

En la estructura del DF traffic, se observa que:
- Las columnas `UpdateTimeUTC` y `UpdateTimeUTC` son de tipo object, es decir, texto. Deben convertirse a date con pd.to_datetime

In [ ]:
# Examinar la estructura de eco
eco.info()
eco.tail(3)

En la estructura del DF eco, se observa que:
- Las columnas `City GDP/capita`, `Unemployment %`, `Population (M)` y `PM2.5 (μg/m³)`  están con formato de texto (object), deben convertirse a float



### 2.2 Renombrar columnas



In [ ]:
# Estandarizar los nombres de las columnas de traffic
#tu código aquí
traffic.rename(columns={'City':'city','Country':'country','UpdateTimeUTC':'update_time_utc',
                        'UpdateTimeUTCWeekAgo':'update_time_utc_week_ago','JamsDelay':'jams_delay','TrafficIndexLive':'traffic_index_live',
                        'JamsLengthInKms':'jams_length_kms','JamsCount':'jams_count','TrafficIndexWeekAgo':'traffic_index_week_ago','TravelTimeLivePer10KmsMins':'travel_time_live_per_10kms_mins','TravelTimeHistoricPer10KmsMins':'travel_time_hist_per_10kms_mins','MinsDelay':'mins_delay'}, inplace=True)
# verificar cambios
traffic.columns


In [ ]:
# Estandarizar los nombres de las columnas de eco
#tu código aquí
eco.rename(columns={'City GDP/capita':'city_gdp_capita','Unemployment %':'unemployment_pct','Country':'country','Year':'year','City':'city','Population (M)':'population_m','PM2.5 (μg/m³)':'pm25'}, inplace=True)

# verificar cambios
eco.columns


### 2.3 Corregir formatos numéricos y de fecha



In [ ]:
# Convertir las columnas de traffic a tipo fecha con pd.to_datetime()
traffic['update_time_utc'] = pd.to_datetime(traffic['update_time_utc'], errors ='coerce', utc=True)
traffic['update_time_utc_week_ago'] = pd.to_datetime(traffic['update_time_utc_week_ago'], errors ='coerce', utc=True)

# verificar el cambio
traffic.info()

In [ ]:
# Limpia separadores y convierte columnas numéricas en eco


eco['city_gdp_capita'] =(eco['city_gdp_capita'].astype("string").str.strip().str.replace(".", "", regex=False).str.replace(",",".",regex=False))
eco['city_gdp_capita']=pd.to_numeric(eco['city_gdp_capita'],errors='coerce')

eco['unemployment_pct']=(eco['unemployment_pct'].astype("string").str.strip().str.replace(",",".").str.replace("%",""))
eco['unemployment_pct']=pd.to_numeric(eco['unemployment_pct'],errors='coerce')

eco['population_m']=(eco['population_m'].astype("string").str.strip().str.replace(",","."))
eco['population_m']=pd.to_numeric(eco['population_m'],errors='coerce')


# Calcula la población total en unidades absolutas (Multiplica * 1000000)
eco['population'] = eco['population_m']*1000000

# verificar el cambio
eco.info()
eco.head(3)

In [ ]:
#Verificar unicidad de Ciudades Y paises en los datasets
#En Traffic
traffic['country'].unique()

In [ ]:
traffic['city'].unique()

In [ ]:
# En eco
eco['country'].unique()

In [ ]:
eco['city'].unique()


---

## 🧩Paso 3: Extraer año y filtrar

Extraer el año permite filtrar la información y trabajar solo con el período más reciente y relevante.

### 3.1 Extraer columna año y filtrar 2024

**🎯Objetivo**
Identificar el año de cada registro y mantener solo los registros del 2024.

**Intrucciones**

- Como el DataFrame `traffic` no tiene una columna de año, utiliza el atributo `.dt.year` sobre su columna de fecha para crear una nueva columna llamada `year`.
- Filtra las filas donde el año sea **2024**.
- Utiliza `.copy()` para crear dos nuevos DataFrames (`traffic_2024` y `eco_2024`) para evitar modificar el dataset original.

In [ ]:
# Extraer el año de las fechas en update_time_utc
traffic['year'] = pd.to_datetime(traffic['update_time_utc']).dt.year

# Verificar cambio
traffic.head(3)

In [ ]:
# Filtra los registros del año 2024
traffic_2024 = traffic[traffic['year']==2024].copy()
eco_2024 = eco[eco['year']==2024].copy()

# Revisar dataframes nuevos
display(traffic_2024.head())
display(eco_2024.head())


<details>
<summary>Haz clic para ver la pista</summary>
Usa ".agg()" para aplicar funciones de promedio. Al final, reinicia el índice para mantener las columnas de la agrupación como variables (no índices).


---

## 🧩Paso 4: Analizar y resumir datos de movilidad

Como el dataset de tráfico contiene **múltiples registros por ciudad**. En esta parte, calcularás los promedios anuales por ciudad para simplificar el análisis y obtener una visión más clara de las tendencias generales.

### 4.1 Calcular promedios de tráfico por ciudad

**🎯Objetivo:**
Obtener una vista consolidada del tráfico promedio por ciudad y año, para analizar patrones generales sin depender de datos diarios.

**Instrucciones**

- Agrupa los datos por `city`, `country` y `year`.
- Calcula el promedio **solo de las métricas de tráfico más relevantes**: como `jams_delay`, `traffic_index_live`, `jams_length_kms`, `jams_count`, `mins_delay`, y tiempos de viaje (`travel_time_live_per_10kms_mins` y `travel_time_hist_per_10kms_mins`).
- Guarda el resultado como `traffic_city_year_2024`, mantén las columnas como variables (no índices).


In [ ]:
# Calcular los  promedios de trafico por ciudad, país y año
traffic_city_year_2024 = (
    traffic_2024.groupby(['country', 'city', 'year'])
    .agg({
        'jams_delay': 'mean',
        'traffic_index_live': 'mean',
        'jams_length_kms': 'mean',
        'jams_count': 'mean',
        'mins_delay': 'mean',
        'travel_time_live_per_10kms_mins': 'mean',
        'travel_time_hist_per_10kms_mins': 'mean'
    })
    .reset_index()
)

# Mostrar resultado
traffic_city_year_2024.head()

### 🧠 **Momento de reflexión**

¡Excelente trabajo hasta aquí!

Ahora que ya tienes los promedios anuales por ciudad, es momento de **observarlos** con atención.

Piensa:

- ¿Cuál crees que tiene el mayor tiempo promedio de tráfico?
- ¿Será una ciudad de **Europa**, de **Latinoamérica** o de **otra región** del mundo?

Para descubrirlo, ejecuta esta línea de código:

`traffic_city_year_2024.sort_values(["jams_delay"], ascending=False)`


🔍 Observa qué ciudad aparece en los primeros lugares.

¿Te sorprenden los resultados? , ¿Coinciden con lo que imaginabas?

In [ ]:
# tu código aquí
traffic_city_year_2024.sort_values(["jams_delay"], ascending=False)

La ciudad con el mayor tiempo promedio de tráfico es Ciudad de México con un promedio de retraso de 2833 min


---

## 🧩Paso 5: Unir movilidad y economía

Combinar datasets te permite analizar cómo se relacionan los indicadores económicos con los de movilidad.

### 5.1 Unir tráfico (tabla principal) con indicadores económicos

**🎯Objetivo:**
Combinar la información de tráfico y economía en un solo DataFrame para analizar cómo las condiciones económicas se relacionan con la movilidad urbana.

**Instrucciones**
- Selecciona solo las **columnas relevantes** de cada dataset (por ejemplo, variables clave de tráfico y de economía).
- Usa `.copy()` al crear subconjuntos para evitar modificar el dataset original.
- Une ambos DataFrames y define como **claves de unión** a `city` y `year`.
- Mantén solo las ciudades y años presentes en ambos datasets.
- Guarda el resultado en una nueva variable llamada `merged` y muestra las primeras 5 filas.


<details>
<summary>Haz clic para ver la pista</summary>
Aplica una unión de tipo "inner" para mantener las ciudades y años presentes en ambos datasets.

In [ ]:
# Seleccionar columnas clave de tráfico y economía
left_cols = ['city','country','year','jams_delay','traffic_index_live',
             'jams_length_kms','jams_count','mins_delay',
             'travel_time_live_per_10kms_mins','travel_time_hist_per_10kms_mins']

right_cols = ['city','year','city_gdp_capita','unemployment_pct','pm25','population']

# Usar .copy() para crear los dos nuevos datasets reducidos
traffic_2024_small = traffic_city_year_2024[(left_cols)].copy()
eco_2024_small = eco_2024[(right_cols)].copy()

# Unir datasets
merged = pd.merge(traffic_2024_small,eco_2024_small, on =['city','year'], how ='inner')

# Mostrar las primeras 5 filas
merged.head()
merged.info()
merged['city'].unique()
merged['country'].unique()


---

## 🧩Paso 6: Visualización y análisis de relaciones

Ahora que tienes un dataset limpio y unificado, es momento de **visualizar patrones**.
Los gráficos te ayudarán a entender cómo se relacionan las variables económicas con las de movilidad urbana.

### 6.1 Visualizar relaciones entre economía y tráfico

**🎯Objetivo:**
Analizar visualmente la distribución y la relación entre indicadores de tráfico y economía en 2024, para identificar posibles patrones o tendencias generales entre ambas variables.

**Instrucciones**
- Usa las librerías `seaborn` y `matplotlib.pyplot` para generar los gráficos.
- Visualiza la distribución del **tráfico** (`jams_delay`) mediante:
    - **Boxplot** → para observar la media, mediana y detectar valores atípicos.
- Visualiza la distribución de la **economía** (`city_gdp_capita`) mediante:
    - **Histograma** → para analizar la forma de la distribución y el valor promedio del PIB per cápita.
- Finalmente, **compara ambas variables**, para observar si existe alguna relación entre ellas, haciendo un solo gráfico de barras donde aparezcan ambos indicadores.
- Recuerda agregar título y etiquetas a los ejes de tus gráficos.
- Observa y comenta los patrones, valores extremos o posibles relaciones que identifiques.

**Tip:** Dentro de los parentesis del boxplot, agrega `showmeans=True` para ver la media en el gráfico.

In [ ]:
# Crear boxplot para observar el comportamiento de los minutos de congestion JamsDelay
# crea tu gráfico
sns.boxplot(data=merged, x='jams_delay',showmeans=True)
# obtener promedio para mostrarlo en título
mean_value = merged['jams_delay'].mean()
plt.title(f'Boxplot de JamsDelay (2024)\nPromedio: {mean_value:.2f}')
plt.show()


In [ ]:
# Crear histograma para ver la distribución de la economía (city_gdp_capita)

merged['city_gdp_capita'].hist(bins=7,figsize=(10,5))
plt.title('Distribución del PIB per cápita por ciudad')

plt.xlabel('PIB per cápita (USD)')

plt.ylabel('Frecuencia')

plt.show()

In [ ]:
# Gráfico de barras para comparar jams_delay y city_gdp_capita por ciudad
#merged.plot( ... , y=['jams_delay', 'city_gdp_capita'])
# Crear figura y primer eje
fig, ax1 = plt.subplots(figsize=(12, 8))

# Configurar el primer eje (jams_delay)
x_pos = range(len(merged))
bars1 = ax1.bar([x - 0.2 for x in x_pos], merged['jams_delay'],
                width=0.4, label='Retraso por congestión',
                color='red', alpha=0.7)

ax1.set_xlabel('Ciudades')
ax1.set_ylabel('Retraso por congestión (jams_delay)', color='red')
ax1.tick_params(axis='y', labelcolor='red')

# Crear segundo eje (city_gdp_capita)
ax2 = ax1.twinx()
bars2 = ax2.bar([x + 0.2 for x in x_pos], merged['city_gdp_capita'],
                width=0.4, label='PIB per cápita',
                color='blue', alpha=0.7)

ax2.set_ylabel('PIB per cápita (USD)', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

# Configurar etiquetas del eje X
ax1.set_xticks(x_pos)
ax1.set_xticklabels(merged['city'], rotation=45, ha='right')

# Título y leyenda
plt.title('Comparación: Congestión vs PIB per cápita por Ciudad')
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))

plt.tight_layout()
plt.xticks(rotation=90)
plt.show()


In [ ]:
# Calcular correlaciones entre todas las variables numéricas
correlation_matrix = merged.select_dtypes(include=['float64', 'int64']).corr()

# Mostrar la matriz
print(correlation_matrix)

**Tip:** Antes del `plt.show()` agrega el código `plt.xticks(rotation=90)` para rotar las etiquetas del eje X en 90 grados.

### 🧠 **Reflexiona**
Excelente trabajo llegando a esta etapa del análisis. Antes de avanzar, revisa tus gráficos, tómate un momento para pensar:

* ¿Las ciudades con mayor PIB per cápita también presentan más congestión?

* ¿O sucede lo contrario, o no existe una relación clara?

Escribe tus comentarios:
Observando el gráfico de "Comparación", se claramente que no hay una correlación exacta entre las variables, ya que algunas ciudades con altos retrasos por con gestión tienen un PIB per Capita menor [sao-paulo, mexico-city, santiago] Sin embargo, en el resto de ciudades si se observa una correlación inversa, mientras más grande es el PIB, menores son los retrasos por congestión.


---

## 🧩Paso 7: Exportar y documentar resultados

En esta etapa final consolidarás todo tu trabajo: guardarás el dataset limpio y crearás un resumen que documente los resultados del proyecto.

### 7.1 Guardar dataset final

**🎯Objetivo:**
Generar un CSV limpio, reproducible y con columnas relevantes para análisis posterior.

**Instrucciones**

- Exporta el DataFrame `merged` con el nombre: `ladb_mobility_economy_2024_clean.csv`
- Usa `index=False` para no incluir el índice.


In [ ]:
# Exporta el dataset final como CSV
merged.to_csv("ladb_mobility_economy_2024_clean.csv", index=False)


---

# 🧾 Resumen ejecutivo

**Contexto & objetivo:**  
La empresa Latin American Development Bank quiere saber si existe una relación entre la movilidad urbana y la productividad económica de las principales ciudades de Latinoamérica.
Para el análisis se trabajan dos bases de datos:
1.Movilidad urbana: TomTom Traffic Index (datos de tráfico en tiempo real).
2.Economía urbana: OECD Cities (PIB per cápita, desempleo y población).

**Cobertura de datos:**  
- Año analizado 2024
- Ciudades analizadas 'buenos-aires', 'belo-horizonte', 'brasilia', 'curitiba',
       'fortaleza', 'porto-alegre', 'recife', 'rio-de-janeiro',
       'salvador', 'sao-paulo', 'santiago', 'bogota', 'mexico-city',
       'lima', 'montevideo'
-Países Analizados 'ARGENTINA', 'BRASIL', 'CHILE', 'COLOMBIA', 'MEXICO', 'PERU', 'URUGUAY

**Metodología (alto nivel):**  
- Se verificaron los tipos de datos de ambos datasets para realizar respectivos cambios a aquellas columnas que no correspondieran a su naturalidad.
- Para el DataFrame de traffic: se convirtieron las columnas correspondientes de **object** a **datetime**

traffic['update_time_utc'] = pd.to_datetime(traffic['update_time_utc'], errors ='coerce', utc=True)

traffic['update_time_utc_week_ago'] = pd.to_datetime(traffic['update_time_utc_week_ago'], errors ='coerce', utc=True).

- Para el DataFrame eco: se estandarizaron y limpiaron las siguientes columnas
- eco['city_gdp_capita'] =(eco['city_gdp_capita'].astype("string").str.strip().str.replace(".", "", regex=False).str.replace(",",".",regex=False))
eco['city_gdp_capita']=pd.to_numeric(eco['city_gdp_capita'],errors='coerce')

eco['unemployment_pct']=(eco['unemployment_pct'].astype("string").str.strip().str.replace(",",".").str.replace("%",""))
eco['unemployment_pct']=pd.to_numeric(eco['unemployment_pct'],errors='coerce')

eco['population_m']=(eco['population_m'].astype("string").str.strip().str.replace(",","."))
eco['population_m']=pd.to_numeric(eco['population_m'],errors='coerce')

- Se tomó la decisión de hacer un merge con "inner", para traer las coincidencias exactas de ambos DataFrames cuyas llaves fueron las columnas ['city'] y ['year'] con el fin de agruparlos y tener un Dataset resumido por año (2024) y por ciudades.
  
- Se empleó un Boxplot de la columna ['jams_delay'] para detectar la media, la mediana y valores atípicos.
- Se emmpleó un Histograma para verificar el tipo de distribución de los datos de la columna ['city_gdp_capita']

**Hallazgos iniciales:**  
 Observando el gráfico de "Comparación", se claramente que no hay una correlación exacta entre las variables, ya que algunas ciudades con altos retrasos por con gestión tienen un PIB per Capita menor [sao-paulo, mexico-city, santiago] Sin embargo, en el resto de ciudades si se observa una correlación inversa, mientras más grande es el PIB, menores son los retrasos por congestión.

- Realizando un análisis profundo, y considerando que las escalas de medición de ambas variables son distintas, se puede inferir que Santiago de chile es la ciudad que su relación de alta congestión vs un bajo PIB es la que más va acorde a la pregunta de negocio. No obstante se observa que Ciudad de México tienes los niveles más altos de congestión, aun teniendo el segundo mejor PIB de las ciudades.

- La ciudad de montevideo debe analizarse más a profundidad, ya que se ven valores mínimos en los retrasos y valores máximos en el PIB

**Recomendaciones**  

Validar fuentes ya que posiblemente halla errores en toma de datos, considerando la ciudad de montevideo, con el mejor PIB y mínimo nivel de congestión.

Teniendo en cuenta la anterior recomendación, si no hubo errores en los datos, estudiar a profundidad el comportamiento de esta ciudad, ¿Hay buena planificación urbana? ¿existe una cultura peatonal y de transporte que evita los trancones y retenes?

En ciudad de México se presenta el mayor nivel de congestión de las 15 ciudades analizadas, pero tiene el mejor segundo PIB. Emplear planes de acción de movilización, invertir en mejoramiento de vías, señalización y cultura, todo esto considerando que es una gran ciudad y tiene un PIB excelente.

Finalemnte para Sao Paulo y Santiago de Chile, ciudades con correlación inversa, alto congestoinamiento y PIB bajo. Realizar campañas de movilización vial para los habitantes. Todo comienza a partir de una buena cultura.
